In [10]:
#imports
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../..')

import numpy as np
import src.viz_utils as viz_utils
import src.utils as utils
import importlib
from src.utils import CanonPart, CanonPartMetadata, get_pointcloud_in_cam_frame, transform_cloud_to_base, remove_outliers
from PIL import Image
import torch
import open3d as o3d
import open3d.visualization.gui as gui
import copy as cp
import pickle
import matplotlib

from src.object_warping import (
    ObjectWarpingSE2Batch,
    ObjectSE2Batch,
    ObjectSE3Batch,
    ObjectWarpingSE3Batch,
    warp_to_pcd,
    warp_to_pcd_se2,
    warp_to_pcd_se3,
    warp_to_pcd_se3_hemisphere,
    PARAM_1,
    ALIGNMENT_PARAM,
    mask_and_cost_batch_pt,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
#load pcl 
root_folder = '/home/rthomp12/fewshot/scripts/experiment_notebooks/z_teapot_on_mug_test_20250115-041705_purple_teapot_pale_mug'
save_name = f'{root_folder}/whole_init_scene_pcls.npz'
scene_pcls = np.load(save_name)
masked_pcls = {k: utils.farthest_point_sample(scene_pcls[k], min(len(scene_pcls[k]), 500))[0] for k in scene_pcls.keys()}



In [12]:
#set objects and parts 

# Mug v Rack
# parent_object = 'tree'
# child_object = 'mug'
# parent_model_file = {'tree': '/home/rthomp12/fewshot/part_based_warp_models/whole_syn_rack_easy_20240412-042732',}['tree']
# child_model_file = {'mug': '/home/rthomp12/fewshot/whole_mug_20240502-182007_6',}['mug']

# if 'tree' not in masked_pcls.keys():
#     masked_pcls = {k: utils.farthest_point_sample(scene_pcls[k], min(len(scene_pcls[k]), 500))[0] for k in scene_pcls.keys()}
#     masked_pcls['tree'] = np.concatenate([masked_pcls['trunk'], masked_pcls['branch']])
#     masked_pcls['mug'] = np.concatenate([masked_pcls['cup'], masked_pcls['handle']])
# else:
#     masked_pcls = {k: utils.farthest_point_sample(scene_pcls[k], min(len(scene_pcls[k]), 1000))[0] for k in scene_pcls.keys()}

# masked_pcls['mug'] = masked_pcls['mug'][masked_pcls["mug"][:,0] - masked_pcls["mug"][:,1]  > -.55]

# # Bowl v Mug

whole = True
parent_object = 'mug'
child_object = 'whole_teapot'

parent_model_file = {'mug': '/home/rthomp12/fewshot/whole_mug_20240502-182007_6',}['mug']
child_model_file = {'whole_teapot': '/home/rthomp12/fewshot/part_based_warp_models/whole_teapot_dict_20241031-012841_5'}['whole_teapot']

parent_object = "mug"


# Mug v Rack
# parent_object = 'mug'
# child_object= 'whole_bowl'


# parent_model_file = {'mug': '/home/rthomp12/fewshot/whole_mug_20240502-182007_6',}['mug']

# child_model_file = {'whole_bowl': '/home/rthomp12/fewshot/part_based_warp_models/whole_bowl_20240426-000022_10'}['whole_bowl']

# parent_object = "mug"



parent_model = CanonPart.from_pickle(parent_model_file)
child_model = CanonPart.from_pickle(child_model_file)

all_names = [parent_object, child_object]

In [13]:
# Warping parameters 
n_angles = 15

PARAM_1 = {"lr": 1e-2, 
           "n_steps": 200,
           "n_samples": 1000, 
           "object_size_reg": 0.03} #.01

inference_kwargs = {
                            "train_latents": True,
                            "train_scales": True,
                            "train_poses": True,
                        }

In [14]:
#relational and variational descriptors

# masked_pcls["whole_teapot"] = masked_pcls["whole_teapot"][masked_pcls["whole_teapot"][:,0] >.6]
# masked_pcls["whole_teapot"] = masked_pcls["whole_teapot"][masked_pcls["whole_teapot"][:,2] <.35]

canon_parent_part_labels = {}
canon_child_part_labels = {}


def get_z_descriptors(part_pcls, part_names):
    part_labels = {part: [] for part in part_names}
    for part in part_names: 
        z_mean = np.mean(part_pcls[part][:,2])
        
        part_labels[part].append(np.where(
                part_pcls[part][:,2] > z_mean,
                np.zeros_like(part_pcls[part][:, 0]),
                np.ones_like(part_pcls[part][:, 0]),
            ))
    return part_labels

def get_x_descriptors(part_pcls, part_names):
    part_labels = {part: [] for part in part_names}
    for part in part_names: 
        z_mean = np.mean(part_pcls[part][:,0])
        
        part_labels[part].append(np.where(
                part_pcls[part][:,0] < z_mean,
                np.zeros_like(part_pcls[part][:, 0]),
                np.ones_like(part_pcls[part][:, 0]),
            ))
    return part_labels



#def fix_pca_descriptors(part_pcls, part_labels, part_pair):
    # part = part_pair[0]
    # other_part = part_pair[1]
    # if np.linalg.norm(np.mean(part_pcls[part][part_labels[part][-1]==1],-1) - np.mean(part_pcls[other_part][part_labels[other_part][-1]==0]-1)) < \
    #    np.linalg.norm(np.mean(part_pcls[part][part_labels[part][-1]==1],-1) - np.mean(part_pcls[other_part][part_labels[other_part][-1]==1]-1)):
    #    part_labels[other_part] = np.logical_not(part_labels[other_part][-1])


canon_parent_part_labels['variational'] = get_z_descriptors({parent_object: parent_model.canonical_pcl}, [parent_object])
#fix_pca_descriptors(parent_canon_pcls, canon_parent_part_labels['variational'], parent_part_names)
canon_child_part_labels['variational'] = get_z_descriptors({child_object: child_model.canonical_pcl}, [child_object])
#fix_pca_descriptors(child_canon_pcls, canon_child_part_labels['variational'], child_part_names)

variational_parent_part_labels = get_z_descriptors(masked_pcls, [parent_object])
variational_child_part_labels = get_x_descriptors(masked_pcls, [child_object])
    
child_reconstructions = {}
child_params = {}
parent_reconstructions = {}
parent_params = {}

In [15]:
#warping

device = 'cuda'

canon = parent_model
print("parent")

warp = ObjectWarpingSE2Batch(
        canon,
        masked_pcls[parent_object],
        device,
        **cp.deepcopy(PARAM_1),
    )
parent_reconstruction, _, parent_params = warp_to_pcd_se2(
    warp, n_angles, n_batches=12, inference_kwargs=inference_kwargs
)



canon = child_model
print("child")

canon_labels = {'variational': canon_child_part_labels['variational'][child_object]}



    # enables optimization incorporating the relational descriptors
cost_function = (
    lambda source, target, canon_part_labels, 
            latent_param, scale_param, initial_latents: mask_and_cost_batch_pt(
        target,
        variational_child_part_labels[child_object],
        source,
        canon_part_labels['variational'],
    ) *.05 +  mask_and_cost_batch_pt(
        target,
        [np.ones_like(variational_child_part_labels[child_object])[0]],
        source,
        [np.ones_like(canon_part_labels['variational'])[0]],
    ) 
)


warp = ObjectWarpingSE3Batch(
        canon,
        masked_pcls[child_object],
        device,
        canon_labels=canon_labels,
        cost_function=cost_function,

        **cp.deepcopy(PARAM_1),
    )
child_reconstruction, _, child_params = warp_to_pcd_se3_hemisphere(
    warp, n_angles, n_batches=12, inference_kwargs=inference_kwargs
)

    

parent
child


In [16]:
all_reconstructions = {child_object: child_reconstruction, parent_object: parent_reconstruction}    
camera = dict(
    eye=dict(x=1.2, y=-1.2, z=.2),
    center=dict(x=.8,y=.5,z=0)
)

viz_utils.show_pcds_plotly({f'reconstructed_{part}': all_reconstructions[part] for part in all_reconstructions.keys()} | 
                           {key: masked_pcls[key] for key in masked_pcls.keys() if len(masked_pcls[key]) > 0}, camera=camera)

In [17]:
save_name = f"{root_folder}/whole_initial_scene_warps"
np.savez(save_name, 
         child_reconstructions=child_reconstruction,
         parent_reconstructions=parent_reconstruction,
         child_params=child_params, 
         parent_params=parent_params,
         )

In [ ]:
# Warping and saving meshes
import trimesh

child_mesh = child_model.to_transformed_mesh(child_params) 
parent_mesh = parent_model.to_transformed_mesh(parent_params) 
child_mesh.export('/home/rthomp12/fewshot/blue_mug_thin_rack_demo/whole_child.obj')
parent_mesh.export('/home/rthomp12/fewshot/blue_mug_thin_rack_demo/whole_parent.obj')

utils.convex_decomposition(child_mesh, '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/demo_child_cd.obj')
utils.convex_decomposition(child_mesh, '/home/rthomp12/fewshot/blue_mug_thin_rack_demo/demo_parent_cd.obj')

INFO - 2025-01-15 04:21:20,350 - generic - executing: /usr/bin/testVHACD /tmp/0_3ixtybqm.obj -resolution 1000000 -depth 20 -concavity 0.0025 -planeDownsampling 4 -convexhullDownsampling 4 -alpha 0.05 -beta 0.05 -gamma 0.00125 -pca 0 -mode 0 -maxNumVerticesPerCH 256 -minVolumePerCH 0.0001 -convexhullApproximation 1 -oclDeviceID 0


INFO - 2025-01-15 04:21:21,731 - generic - 
[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 0% : ComputingBoun[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 100% : ComputingBounds
[CREATE_RAYCAST_MESH                     ] : 20% : 0% : Building RaycastMe[CREATE_RAYCAST_MESH                     ] : 20% : 100% : RaycastMesh completed
[VOXELIZING_INPUT_MESH                   ] : 30% : 0% : Voxelizing Input Mes[VOXELIZING_INPUT_MESH                   ] : 30% : 100% : Voxelization complete
[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 0% : Build initial ConvexHu[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 100% : Initial ConvexHull complete
[PERFORMING_DECOMPOSITION                ] : 50% : 0% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION                ] : 50% : 0% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION                ] : 50% : 2% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION    

/home/rthomp12/fewshot/blue_mug_thin_rack_demo/demo_child_cd.obj


INFO - 2025-01-15 04:21:23,187 - generic - 
[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 0% : ComputingBoun[COMPUTE_BOUNDS_OF_INPUT_MESH            ] : 0% : 100% : ComputingBounds
[CREATE_RAYCAST_MESH                     ] : 20% : 0% : Building RaycastMe[CREATE_RAYCAST_MESH                     ] : 20% : 100% : RaycastMesh completed
[VOXELIZING_INPUT_MESH                   ] : 30% : 0% : Voxelizing Input Mes[VOXELIZING_INPUT_MESH                   ] : 30% : 100% : Voxelization complete
[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 0% : Build initial ConvexHu[BUILD_INITIAL_CONVEX_HULL               ] : 40% : 100% : Initial ConvexHull complete
[PERFORMING_DECOMPOSITION                ] : 50% : 0% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION                ] : 50% : 0% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION                ] : 50% : 2% : Performing recursive decomposition of convex hul[PERFORMING_DECOMPOSITION    

/home/rthomp12/fewshot/blue_mug_thin_rack_demo/demo_parent_cd.obj


[<trimesh.Trimesh(vertices.shape=(32, 3), faces.shape=(60, 3))>,
 <trimesh.Trimesh(vertices.shape=(64, 3), faces.shape=(124, 3))>,
 <trimesh.Trimesh(vertices.shape=(24, 3), faces.shape=(44, 3))>,
 <trimesh.Trimesh(vertices.shape=(64, 3), faces.shape=(124, 3))>,
 <trimesh.Trimesh(vertices.shape=(57, 3), faces.shape=(110, 3))>,
 <trimesh.Trimesh(vertices.shape=(36, 3), faces.shape=(68, 3))>,
 <trimesh.Trimesh(vertices.shape=(48, 3), faces.shape=(92, 3))>,
 <trimesh.Trimesh(vertices.shape=(42, 3), faces.shape=(80, 3))>,
 <trimesh.Trimesh(vertices.shape=(25, 3), faces.shape=(46, 3))>,
 <trimesh.Trimesh(vertices.shape=(44, 3), faces.shape=(84, 3))>,
 <trimesh.Trimesh(vertices.shape=(32, 3), faces.shape=(60, 3))>,
 <trimesh.Trimesh(vertices.shape=(36, 3), faces.shape=(68, 3))>,
 <trimesh.Trimesh(vertices.shape=(19, 3), faces.shape=(34, 3))>,
 <trimesh.Trimesh(vertices.shape=(27, 3), faces.shape=(50, 3))>,
 <trimesh.Trimesh(vertices.shape=(45, 3), faces.shape=(86, 3))>,
 <trimesh.Trimesh(vert

: 